In [ ]:
import os
import re
import sys
import random
import platform
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report,confusion_matrix

In [ ]:
SEED =42
DATASET ="cifar10"
BATCH_SIZE=64
EPOCHS=50
LEARNİNG_RATE =1e-3
VAL_SPLIT = 0.2
RESULT_DIR = "results"
FIG_DIR = os.path.join(RESULT_DIR,"figs")
MODEL_DIR = os.path.join(RESULT_DIR,"models")
os.makedirs(FIG_DIR,exist_ok=True)
os.makedirs(MODEL_DIR,exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

In [ ]:
CLASS_NAMES = {
    "mnist": [str(i) for i in range(10)],
    "fashion_mnist": ["tisort", "pantolon", "kazak", "elbise", "mont",
                      "sandalet", "gomlek", "spor_ayakkabi", "canta", "bot"],
    "cifar10": ["ucak", "otomobil", "kus", "kedi", "geyik",
                "kopek", "kurbaga", "at", "gemi", "kamyon"],
    "cifar100": [f"sinif_{i}" for i in range(100)],
}


In [ ]:
def load_data():
  """hazır veri setini yükler:(train_ds,val_ds,test_ds)"""
  loaders ={
      "mnist":keras.datasets.mnist.load_data,
      "fashion_mnist":keras.datasets.fashion_mnist.load_data,
      "cifar10":keras.datasets.cifar10.load_data,
      "cifar100":keras.datasets.cifar100.load_data,
  }
  (x_train,y_train),(x_test,y_test)= loaders[DATASET]()
  y_train,y_test=y_train.flatten() ,y_test.flatten()

  if x_train.ndim == 3:
    x_train, x_test = x_train[..., None], x_test[..., None]

  print(f"Veri seti: {DATASET} | Eğitim: {x_train.shape} | Test: {x_test.shape}")

  n_val = int(len(x_train) * VAL_SPLIT)
  idx = np.random.permutation(len(x_train))
  val_idx, tr_idx = idx[:n_val], idx[n_val:]

  def make_ds(x, y, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(10_000, seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

  return (make_ds(x_train[tr_idx], y_train[tr_idx], True),
          make_ds(x_train[val_idx], y_train[val_idx], False),
          make_ds(x_test, y_test, False),
          CLASS_NAMES[DATASET])

In [ ]:
def conv_block(x,filters,dropout):
  for _ in range(2):
    x=layers.Conv2D(filters,3,padding="same",use_bias = False)(x)
    x=layers.BatchNormalization()(x)
    x=layers.Activation("relu")(x)
  x=layers.MaxPooling2D()(x)
  return layers.Dropout(dropout)(x)

def build_model(input_shape,num_classes):
  inputs = keras.Input(shape=input_shape)

  x=inputs
  if DATASET != "mnist":
      x=layers.RandomFlip("horizontal", seed=SEED)(x)
  x=layers.RandomRotation(0.1,seed=SEED)(x)
  x=layers.RandomZoom(0.1,seed=SEED)(x)

  x=layers.Rescaling(1/255)(x)
  x=conv_block(x,32,0.2)
  x=conv_block(x,64,0.3)
  x=conv_block(x,128,0.4)
  x=layers.GlobalAveragePooling2D()(x)
  x=layers.Dense(256,activation="relu")(x)
  x=layers.Dropout(0.5)(x)
  x=layers.Dense(128,activation="relu")(x)
  outputs=layers.Dense(num_classes,activation="softmax")(x)

  return keras.Model(inputs,outputs)

In [ ]:
def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric, title in zip(axes, ["loss", "accuracy"], ["Kayıp", "Doğruluk"]):
        ax.plot(history.history[metric], label="Eğitim")
        ax.plot(history.history[f"val_{metric}"], label="Doğrulama")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "egitim_egrileri.png"), dpi=300)
    plt.show()


In [ ]:
train_ds, val_ds, test_ds, class_names = load_data()


for X_batch, y_batch in train_ds.take(1):
    input_shape = X_batch.shape[1:]
    num_classes = len(class_names)

model = build_model(input_shape, num_classes)
model.compile(
    optimizer=keras.optimizers.Adam(LEARNİNG_RATE),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    verbose=2
)


plot_history(history)


import numpy as np


X_test_list = []
y_true_list = []
for X_batch, y_batch in test_ds:
    X_test_list.append(X_batch.numpy())
    y_true_list.append(y_batch.numpy())
X_test = np.concatenate(X_test_list, axis=0)
y_true = np.concatenate(y_true_list, axis=0)

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)


def plot_confusion(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.savefig(os.path.join(FIG_DIR, "confusion_matrix.png"), dpi=300)
    plt.show()

plot_confusion(y_true, y_pred, class_names=class_names)

Veri seti: cifar10 | Eğitim: (50000, 32, 32, 3) | Test: (10000, 32, 32, 3)
Epoch 1/50
625/625 - 433s - 692ms/step - accuracy: 0.3616 - loss: 1.7073 - val_accuracy: 0.4758 - val_loss: 1.4494
Epoch 2/50
625/625 - 442s - 707ms/step - accuracy: 0.4869 - loss: 1.4032 - val_accuracy: 0.4537 - val_loss: 1.6798
Epoch 3/50
625/625 - 414s - 663ms/step - accuracy: 0.5393 - loss: 1.2738 - val_accuracy: 0.5330 - val_loss: 1.3969
Epoch 4/50
625/625 - 429s - 686ms/step - accuracy: 0.5712 - loss: 1.1995 - val_accuracy: 0.6040 - val_loss: 1.1083
Epoch 5/50
